# CodeMixBridge: Kannada-English Code-Mixed Language Conversion
### Fine-Tuning Google Gemma 4 on IndicCMix via LoRA on Google Colab (T4 GPU)

**Hackathon:** Hacktoberfest Hack Day — **Problem Statement 3A**

#### Project Overview:
People frequently communicate by mixing English with Kannada in informal chats (*Kanglish*). Generic large language models often struggle with informal transliterations, producing unnatural literal translations, confusing scripts, or generating verbose explanations.

**CodeMixBridge** fine-tunes Google's **Gemma 4 (E2B)** model on the **AI4Bharat IndicCMix** Kannada dataset to master three core conversions:
1. **English $\rightarrow$ Casual Romanized Kannada (Kanglish)**
2. **English $\rightarrow$ Native Script Code-Mixed Kannada**
3. **English $\rightarrow$ Full Native Kannada Script**
(plus bidirectional reverse translation tasks).

#### Two-Model Architecture Protocol:
- **Model A (Generic Gemma):** Base `unsloth/gemma-4-E2B-it`, no LoRA adapter, standard prompt, deterministic generation.
- **Model B (Fine-Tuned Gemma):** Identical base model, active Kannada LoRA adapter, identical standard prompt, identical generation parameters.

---

## 1. Environment & GPU Verification
Verify that a GPU runtime is selected: **Runtime $\rightarrow$ Change runtime type $\rightarrow$ T4 GPU**.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU detected! Please navigate to Runtime -> Change runtime type -> T4 GPU and restart.")
dev_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ GPU Ready: {dev_name} ({vram:.2f} GB total VRAM)")

## 2. Dependency Installation
Install Unsloth, compatible Transformers, PEFT, TRL, SacreBLEU, and Gradio.

In [ ]:
%%capture
import os
if "COLAB_" in "".join(os.environ.keys()):
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer pyarrow
    !pip install --upgrade "unsloth>=2026.10.7" "unsloth_zoo>=2026.10.7" bitsandbytes accelerate peft trl triton
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install "huggingface_hub>=1.5.0,<2.0"
    !pip install --no-deps --upgrade timm
    !pip install "gradio>=6,<7" pandas sacrebleu tabulate
    !pip install --upgrade --force-reinstall --no-cache-dir --no-deps "unsloth>=2026.10.7" "unsloth_zoo>=2026.10.7"

# This must happen before importing transformers, TRL, PEFT, datasets, or huggingface_hub.
import unsloth

## 3. Hugging Face Authentication
`ai4bharat/IndicCMix` is a gated dataset on Hugging Face.
1. Visit https://huggingface.co/datasets/ai4bharat/IndicCMix and accept dataset terms.
2. Create an access token at https://huggingface.co/settings/tokens.
3. Add your token in Colab Secrets as `HF_TOKEN`, or enter it when prompted below.

In [ ]:
import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✅ Logged in to Hugging Face successfully via HF_TOKEN.")
else:
    print("Please log in interactively:")
    login()

## 4. Dataset Loading (AI4Bharat IndicCMix Kannada)
We load `ka.parquet` and remove audio columns to extract the text data.

In [ ]:
from datasets import load_dataset

print("Loading Kannada dataset from ai4bharat/IndicCMix (ka.parquet)...")
raw_dataset = load_dataset("ai4bharat/IndicCMix", data_files="ka.parquet", split="train")
if "audio" in raw_dataset.column_names:
    raw_dataset = raw_dataset.remove_columns(["audio"])

print(f"✅ Successfully loaded {len(raw_dataset)} raw records.")
print("Columns present:", raw_dataset.column_names)

## 5. Dataset Inspection & Quality Check
Display first 5 raw records to inspect the four parallel representations.

In [ ]:
for i in range(min(5, len(raw_dataset))):
    item = raw_dataset[i]
    print(f"--- RECORD {i+1} ---")
    print("English:           ", item['english'])
    print("Romanized Casual:  ", item['romanized_casual'])
    print("Native Code-Mixed: ", item['native_script_codemixed'])
    print("Full Native Script:", item['full_native_script'])
    print()

## 6. Unicode Normalization & Group-Aware Leakage-Free Splitting
To prevent data leakage, we group records by normalized English source text, assigning a deterministic `source_id`. Split: **70% Train, 15% Validation, 15% Held-Out Test** (Seed: `3407`).

In [ ]:
import unicodedata, hashlib, random, json

def normalize_text(text):
    if not text:
        return ""
    norm = unicodedata.normalize("NFC", str(text)).replace("\u00a0", " ")
    lines = [" ".join(line.split()) for line in norm.splitlines()]
    return "\n".join(lines).strip()

def generate_source_id(text):
    norm = normalize_text(text).lower()
    return f"src_{hashlib.sha256(norm.encode('utf-8')).hexdigest()[:12]}"

# Clean and group records
cleaned_records = []
seen_keys = set()
for row in raw_dataset:
    eng = normalize_text(row.get("english", ""))
    cmix = normalize_text(row.get("native_script_codemixed", ""))
    native = normalize_text(row.get("full_native_script", ""))
    rom = normalize_text(row.get("romanized_casual", ""))
    if not eng or not cmix or not native or not rom:
        continue
    key = (eng.lower(), rom.lower())
    if key in seen_keys:
        continue
    seen_keys.add(key)
    cleaned_records.append({
        "source_id": generate_source_id(eng),
        "english": eng,
        "native_script_codemixed": cmix,
        "full_native_script": native,
        "romanized_casual": rom,
    })

# Group by source_id
groups = {}
for r in cleaned_records:
    groups.setdefault(r["source_id"], []).append(r)

unique_sids = sorted(list(groups.keys()))
rng = random.Random(3407)
rng.shuffle(unique_sids)

n_train = int(len(unique_sids) * 0.70)
n_val = int(len(unique_sids) * 0.15)

train_sids = set(unique_sids[:n_train])
val_sids = set(unique_sids[n_train:n_train+n_val])
test_sids = set(unique_sids[n_train+n_val:])

# Verify zero leakage
assert len(train_sids.intersection(test_sids)) == 0, "Train/Test leakage!"
assert len(val_sids.intersection(test_sids)) == 0, "Val/Test leakage!"

train_recs = [r for sid in train_sids for r in groups[sid]]
val_recs = [r for sid in val_sids for r in groups[sid]]
test_recs = [r for sid in test_sids for r in groups[sid]]

print(f"Partition Summary:\n- Train: {len(train_recs)} records\n- Val:   {len(val_recs)} records\n- Test:  {len(test_recs)} records (Zero Leakage Verified)")

## 7. Supervised Task Example Construction
We expand records into standard prompt-response pairs for the 3 primary tasks:
1. `english_to_romanized`
2. `english_to_codemixed`
3. `english_to_native`

In [ ]:
TASK_INSTRUCTIONS = {
    "english_to_romanized": (
        "Convert the following English message into natural casual Romanized Kannada (Kanglish).\n"
        "Preserve meaning, names, numbers, dates, and negation.\n"
        "Return only the converted message."
    ),
    "english_to_codemixed": (
        "Convert the following English message into natural Kannada script code-mixed text.\n"
        "Preserve meaning, names, numbers, dates, and negation.\n"
        "Return only the converted message."
    ),
    "english_to_native": (
        "Convert the following English message into clear, formal native Kannada script.\n"
        "Preserve meaning, names, numbers, dates, and negation.\n"
        "Return only the converted message."
    ),
}

def make_task_examples(records):
    examples = []
    for r in records:
        sid = r["source_id"]
        eng = r["english"]
        # Task 1
        examples.append({
            "source_id": sid, "task": "english_to_romanized",
            "input": f"{TASK_INSTRUCTIONS['english_to_romanized']}\n\nText:\n{eng}",
            "output": r["romanized_casual"]
        })
        # Task 2
        examples.append({
            "source_id": sid, "task": "english_to_codemixed",
            "input": f"{TASK_INSTRUCTIONS['english_to_codemixed']}\n\nText:\n{eng}",
            "output": r["native_script_codemixed"]
        })
        # Task 3
        examples.append({
            "source_id": sid, "task": "english_to_native",
            "input": f"{TASK_INSTRUCTIONS['english_to_native']}\n\nText:\n{eng}",
            "output": r["full_native_script"]
        })
    return examples

train_examples = make_task_examples(train_recs)
val_examples = make_task_examples(val_recs)
test_examples = make_task_examples(test_recs)

print(f"Expanded Task Counts: {len(train_examples)} Train, {len(val_examples)} Val, {len(test_examples)} Test")

## 8. Load Base Gemma 4 Model
We load Google's `unsloth/gemma-4-E2B-it` in 4-bit precision via Unsloth.

In [ ]:
from unsloth import FastModel

MAX_SEQ_LENGTH = 1024
MODEL_NAME = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit"

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    text_only=True,
    offload_layers="auto",
)
print(f"✅ Successfully loaded base model: {MODEL_NAME}")

## 9. Baseline Inference Function
Deterministic text generation helper.

In [ ]:
import torch

def generate(user_text, max_new_tokens=128):
    messages = [{"role": "user", "content": [{"type": "text", "text": user_text}]}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Test base model response
print("Sample Base Model Output:")
print(generate("Convert to Kannada: Good morning friend"))

## 10. Baseline Evaluation of Generic Gemma (Model A)
Evaluate un-fine-tuned Gemma on held-out test examples using SacreBLEU chrF score.

In [ ]:
import sacrebleu, pandas as pd

# Rebuild the task examples if the runtime has the split but this cell was skipped.
if "test_examples" not in globals():
    if "test_recs" in globals() and "make_task_examples" in globals():
        test_examples = make_task_examples(test_recs)
    else:
        raise RuntimeError("Run Sections 4-7 first to load, clean, split, and expand the dataset.")

# Evaluate 30 held-out task examples for benchmark timing.
EVAL_SUBSET = test_examples[:30]

baseline_rows = []
print("Evaluating Generic Gemma Baseline on held-out test data...")
for i, ex in enumerate(EVAL_SUBSET, start=1):
    pred = generate(ex["input"], max_new_tokens=96)
    chrf = sacrebleu.sentence_chrf(pred, [ex["output"]]).score
    baseline_rows.append({
        "source_id": ex["source_id"],
        "task": ex["task"],
        "input": ex["input"],
        "reference": ex["output"],
        "generic_prediction": pred,
        "chrf": chrf,
    })
    print(f"\rEvaluated {i}/{len(EVAL_SUBSET)}...", end="")

baseline_df = pd.DataFrame(baseline_rows)
base_chrf = baseline_df["chrf"].mean()
print(f"\n\nGeneric Gemma Baseline Mean chrF: {base_chrf:.2f}")
baseline_df[["task", "reference", "generic_prediction", "chrf"]].head(5)

## 11. LoRA Configuration
Apply Low-Rank Adaptation (LoRA) to Gemma 4 language layers ($r=16$, $\alpha=16$).

In [ ]:
LORA_RANK = 16
TRAIN_STEPS = 1000
LEARNING_RATE = 2e-4

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)
print("✅ LoRA adapter wrapped on Gemma 4.")

## 12. Format SFT Dataset (Response-Only Loss Masking)
Apply the Gemma 4 chat template and mask prompt tokens so the loss is evaluated only on assistant Kannada responses.

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def format_to_chat(ex):
    conv = [
        {"role": "user", "content": [{"type": "text", "text": ex["input"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": ex["output"]}]}
    ]
    text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
    return {"text": text.removeprefix("<bos>")}

sft_train_dataset = Dataset.from_list(train_examples).map(format_to_chat)
print("Sample Formatted Training Record:")
print(sft_train_dataset[0]["text"])

## 13. Fine-Tuning Execution
Train with TRL's `SFTTrainer` on the Colab T4 GPU.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=sft_train_dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.05,
        max_steps=TRAIN_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",
        output_dir="gemma4_kannada_checkpoints",
    ),
)

# Mask user prompt tokens
trainer = train_on_responses_only(trainer)

print("Starting LoRA training on T4 GPU...")
train_stats = trainer.train()
print(f"\n✅ Training complete in {train_stats.metrics['train_runtime']/60:.2f} minutes!")

## 14. Save Fine-Tuned LoRA Adapters

In [ ]:
ADAPTER_DIR = "gemma_4_kannada_lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
!zip -qr gemma_4_kannada_lora.zip gemma_4_kannada_lora
print(f"✅ Adapters saved to '{ADAPTER_DIR}' and compressed to 'gemma_4_kannada_lora.zip'.")

## 15. Comparative Evaluation (Generic Gemma vs Fine-Tuned Gemma)
Evaluate Fine-Tuned Gemma on the exact same held-out test subset and compare chrF metrics.

In [ ]:
import pandas as pd
import sacrebleu

if "test_examples" not in globals():
    raise RuntimeError("Run the dataset-preparation cell first to recreate test_examples.")
if not hasattr(model, "disable_adapter"):
    raise RuntimeError("The model has no LoRA adapter. Run Sections 11-13 before this comparison.")

# Deterministic balanced subset: 10 held-out examples from each task.
task_names = ["english_to_romanized", "english_to_codemixed", "english_to_native"]
EVAL_SUBSET = [
    ex for task in task_names
    for ex in [item for item in test_examples if item["task"] == task][:10]
]

# Model A: temporarily disable LoRA without changing the trained adapter.
baseline_rows = []
print("Evaluating Generic Gemma (Model A)...")
with model.disable_adapter():
    for i, ex in enumerate(EVAL_SUBSET, start=1):
        pred = generate(ex["input"], max_new_tokens=96)
        chrf = sacrebleu.sentence_chrf(pred, [ex["output"]]).score
        baseline_rows.append({
            "source_id": ex["source_id"], "task": ex["task"],
            "reference": ex["output"], "generic_prediction": pred, "chrf": chrf,
        })
        print(f"\rModel A: {i}/{len(EVAL_SUBSET)}", end="", flush=True)

baseline_df = pd.DataFrame(baseline_rows)
base_chrf = baseline_df["chrf"].mean()
print()

finetuned_rows = []
adapter_names = list(getattr(model, "peft_config", {}).keys())
if not adapter_names:
    raise RuntimeError("No trained LoRA adapter is attached to the model.")
model.set_adapter(adapter_names[0], inference_mode=True)
print(f"Explicitly activated adapter for Model B: {adapter_names[0]}")
print("Evaluating Fine-Tuned Gemma (Model B) on test data...")
for i, ex in enumerate(EVAL_SUBSET, start=1):
    pred = generate(ex["input"], max_new_tokens=96)
    chrf = sacrebleu.sentence_chrf(pred, [ex["output"]]).score
    finetuned_rows.append({
        "source_id": ex["source_id"],
        "task": ex["task"],
        "finetuned_prediction": pred,
        "finetuned_chrf": chrf,
    })
    print(f"\rEvaluated {i}/{len(EVAL_SUBSET)}...", end="")

finetuned_df = pd.DataFrame(finetuned_rows)
tuned_chrf = finetuned_df["finetuned_chrf"].mean()

comparison = pd.DataFrame({
    "Model Configuration": ["Model A: Generic Gemma (Base)", "Model B: Fine-Tuned Gemma (LoRA)"],
    "Mean chrF Score": [round(base_chrf, 2), round(tuned_chrf, 2)],
})
comparison["Net Improvement (points)"] = comparison["Mean chrF Score"] - round(base_chrf, 2)

print("\n\n=== OFFICIAL HACKATHON BENCHMARK RESULTS ===")
print(comparison.to_string(index=False))
print("=" * 44)

# Verify that enabling LoRA actually changed model output.
prediction_check = baseline_df.merge(finetuned_df, on=["source_id", "task"])
changed_count = (prediction_check["generic_prediction"] != prediction_check["finetuned_prediction"]).sum()
print(f"Predictions changed by LoRA: {changed_count}/{len(prediction_check)}")
if changed_count == 0:
    print("WARNING: Model A and Model B produced identical outputs. Verify that training completed and the LoRA adapter is active.")

per_task = prediction_check.groupby("task").agg(
    generic_chrf=("chrf", "mean"),
    finetuned_chrf=("finetuned_chrf", "mean"),
).round(2)
per_task["improvement"] = (per_task["finetuned_chrf"] - per_task["generic_chrf"]).round(2)
display(per_task)

## 16. Qualitative Side-by-Side Comparison

In [ ]:
side_by_side = baseline_df[["task", "reference", "generic_prediction"]].copy()
side_by_side["finetuned_prediction"] = finetuned_df["finetuned_prediction"]
side_by_side.head(8)

## 17. Export Benchmark Metrics to CSV
Export the real measured results to `metrics.csv` for the Gradio app and documentation.

In [ ]:
# Compute per-task breakdown
merged = baseline_df.merge(finetuned_df, on=["source_id", "task"])

def task_mean(df, task, col):
    sub = df[df["task"] == task]
    return round(sub[col].mean(), 2) if len(sub) > 0 else 0.0

metrics_export = pd.DataFrame([
    {
        "Model": "Generic Gemma (Base)",
        "Romanized chrF": task_mean(merged, "english_to_romanized", "chrf"),
        "Code-mixed chrF": task_mean(merged, "english_to_codemixed", "chrf"),
        "Native chrF": task_mean(merged, "english_to_native", "chrf"),
        "Overall chrF": round(base_chrf, 2),
    },
    {
        "Model": "Fine-Tuned Gemma (LoRA)",
        "Romanized chrF": task_mean(merged, "english_to_romanized", "finetuned_chrf"),
        "Code-mixed chrF": task_mean(merged, "english_to_codemixed", "finetuned_chrf"),
        "Native chrF": task_mean(merged, "english_to_native", "finetuned_chrf"),
        "Overall chrF": round(tuned_chrf, 2),
    }
])
metrics_export.to_csv("metrics.csv", index=False)
print("✅ Saved metrics to metrics.csv:")
print(metrics_export.to_string(index=False))

## 18. Launch CodeMixBridge Dual-Model Web Application
Launches the live side-by-side interface comparing **Generic Gemma vs Fine-Tuned Gemma** using adapter toggling (`with model.disable_adapter():`).

In [ ]:
import gradio as gr

gr.close_all()

def generate_plain(prompt, max_new_tokens=128):
    try:
        with model.disable_adapter():
            return generate(prompt, max_new_tokens=max_new_tokens)
    except Exception as e:
        return f"(Plain model error: {e})"

def run_conversion(input_text, task_label):
    if not input_text.strip():
        raise gr.Error("Please enter a message.")
    
    task_map = {
        "English -> Romanized Kannada (Casual Chat)": "english_to_romanized",
        "English -> Kannada Code-Mixed (Native Script)": "english_to_codemixed",
        "English -> Full Kannada (Formal Script)": "english_to_native",
    }
    task_key = task_map.get(task_label, "english_to_romanized")
    prompt = f"{TASK_INSTRUCTIONS[task_key]}\n\nText:\n{input_text.strip()}"
    
    generic_res = generate_plain(prompt, max_new_tokens=96)
    finetuned_res = generate(prompt, max_new_tokens=96)
    return generic_res, finetuned_res

with gr.Blocks(title="CodeMixBridge — Gemma 4 Kannada AI") as demo:
    gr.Markdown("# 🌉 CodeMixBridge: Kannada-English AI\n### Powered by Google Gemma 4 & LoRA Fine-Tuning")
    with gr.Row():
        with gr.Column(scale=4):
            task_sel = gr.Dropdown(
                choices=[
                    "English -> Romanized Kannada (Casual Chat)",
                    "English -> Kannada Code-Mixed (Native Script)",
                    "English -> Full Kannada (Formal Script)"
                ],
                value="English -> Romanized Kannada (Casual Chat)",
                label="Select Conversion Task"
            )
            input_box = gr.Textbox(lines=3, label="Input Message", placeholder="Enter English message...")
            submit_btn = gr.Button("🚀 Run Dual Model Conversion", variant="primary")
            
            gr.Examples([
                ["Our internal lab exams start tomorrow at 9 AM, please arrive on time.", "English -> Romanized Kannada (Casual Chat)"],
                ["I am submitting the assignment in the computer lab today.", "English -> Kannada Code-Mixed (Native Script)"],
                ["The college library will remain closed on Sunday for maintenance.", "English -> Full Kannada (Formal Script)"],
            ], inputs=[input_box, task_sel])
            
        with gr.Column(scale=6):
            generic_out = gr.Textbox(label="MODEL A: Generic Gemma (Base)", lines=4)
            finetuned_out = gr.Textbox(label="MODEL B: Fine-Tuned Gemma (LoRA)", lines=4)
            
    submit_btn.click(run_conversion, inputs=[input_box, task_sel], outputs=[generic_out, finetuned_out])

demo.launch(share=True)